# Homework 3
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

### Module Imports / Importación de Módulos
This section imports necessary libraries for numerical analysis, data manipulation, and visualization.

En esta sección se importan las bibliotecas necesarias para el análisis numérico, la manipulación de datos y la visualización.

### Importación de módulos
Se importan bibliotecas necesarias para el análisis numérico, la manipulación de datos y la visualización.

In [27]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.spatial import distance
from scipy.stats import wrapcauchy, levy_stable
import math


### Definition of Classes and Helper Functions / Definición de Clases y Funciones Auxiliares
This section defines classes and utility functions to facilitate trajectory computations and distance calculations in random motion models.

En esta sección se definen clases y funciones auxiliares para facilitar los cálculos de trayectorias y distancias en modelos de movimiento aleatorio.

# Functions

### Definición de Clases y Funciones Auxiliares
Se definen clases y funciones que facilitan el cálculo de trayectorias y distancias en los modelos de movimiento.

### Simulation of Random Walks / Simulación de Caminatas Aleatorias
Different random walk simulations are implemented, including Brownian Motion and Lévy Flights.

Se implementan distintas simulaciones de caminatas aleatorias, como el Movimiento Browniano y los Vuelos de Lévy.

In [28]:
# Nota: Esta clase la importaremos junto con el segundo bloque de modulos
################# http://www.pygame.org/wiki/2DVectorClass ##################
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)
     # Método para convertir el vector en una tupla
    def to_tuple(self):
        return (self.x, self.y)

In [29]:
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=5, s_pos=[0,0]):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
    Returns:
        BM_2d_df:
    """
    # Init velocity vector
    velocity =Vec2d(speed,0)
    
    # Init DF
    BM_2d_df = pd.DataFrame(columns=['x_pos','y_pos'])    
    # Add initial position
    temp_df = pd.DataFrame([{'x_pos':s_pos[0], 'y_pos':s_pos[1]}])    
    BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
    
    # Generate the trajectory
    for i in range(n_steps-1):        
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        velocity = velocity.rotated(turn_angle)
    
        temp_df = pd.DataFrame([{'x_pos':BM_2d_df.x_pos[i]+velocity.x, 'y_pos':BM_2d_df.y_pos[i]+velocity.y}])    
        BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
        
    return BM_2d_df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_2d(n_steps=1000, speed=5, s_pos=[0,0],c =0.5):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
    Returns:
        BM_2d_df:
    """
    # Init velocity vector
    velocity =Vec2d(speed,0)
    
    # Init DF
    BM_2d_df = pd.DataFrame(columns=['x_pos','y_pos'])    
    # Add initial position
    temp_df = pd.DataFrame([{'x_pos':s_pos[0], 'y_pos':s_pos[1]}])    
    BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
    
    # Generate the trajectory
    for i in range(n_steps-1):        
        turn_angle =   wrapcauchy.rvs(c)   
        velocity = velocity.rotated(turn_angle)
    
        temp_df = pd.DataFrame([{'x_pos':BM_2d_df.iloc[i]['x_pos'] + velocity.x, 'y_pos':BM_2d_df.y_pos[i]+velocity.y}])    
        BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
        
    return BM_2d_df

#####################################################################################
# Correlated Random Walk 
#####################################################################################
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c = 0.5):
    pos = Vec2d(0, 0)
    trajectory = [pos.to_tuple()]
    angle = 0  # Ángulo inicial en radianes
    for i in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))  # Tamaño del paso con Lévy
        delta_angle = wrapcauchy.rvs(c)  # Generar un ángulo con distribución de Cauchy
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    print("Primeros 5 puntos de la trayectoria:", trajectory[:5])  # Verifica si hay datos

    x, y = zip(*trajectory)
    z = np.linspace(0, 1, len(x))  # Crear un eje Z para la visualización 3D
    
    print("Cantidad de puntos generados:", len(x))  # Debe ser n_steps + 1

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', name='Lévy Flight'))
    fig.show()
    
    #devolver DataFrame para análisis externo
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    df["step_length"] = np.sqrt(df["x_pos"].diff()**2 + df["y_pos"].diff()**2)

    return df


### Path Length Analysis / Análisis de la Longitud de Trayectoria
This section compares the total distance traveled for different random walk models, evaluating the cumulative path length over time.

Se compara la distancia total recorrida en diferentes modelos de caminatas aleatorias, evaluando la longitud acumulativa del camino a lo largo del tiempo.

### Simulación de Caminatas Aleatorias
Aquí se implementan distintas simulaciones de caminatas aleatorias, como el movimiento Browniano y el vuelo de Lévy.

# Activity 1: Path length - BM1 vs BM2 vs CRW

### Mean Squared Displacement (MSD) Analysis / Análisis del Desplazamiento Cuadrático Medio (MSD)
The MSD is computed to analyze the relationship between time and displacement in each type of random walk.

Se calcula el MSD para analizar la relación entre el tiempo y la distancia recorrida en cada tipo de caminata aleatoria.

In [30]:
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import distance

###############
# Path Length #
###############
def path_length(trajectory):
    # Get the Euclidean Distance
    distances = np.array([distance.euclidean(trajectory.iloc[i-1], trajectory.iloc[i]) for i in range(1, trajectory.shape[0])])
    # Get the Cumulative Sum of the steps
    return np.cumsum(distances)

n_steps = 1000

# Definir las configuraciones de cada tipo de caminata
walks = {
    "BM_3": bm_2d(n_steps, speed=3),
    "BM_6": bm_2d(n_steps, speed=6),
    "CRW_5": rw_2d(n_steps, speed=5),
    "CRW_6": rw_2d(n_steps, speed=6),
    "Levy_1": levy_flight(n_steps, alpha=1),
    "Levy_07": levy_flight(n_steps, alpha=0.7)
}

# Imprimir qué claves están en walks para depuración
print("walks keys:", list(walks.keys()))

# Filtrar solo las caminatas que no son None
path_lengths = {key: path_length(df) for key, df in walks.items() if df is not None}

# Imprimir claves en path_lengths para verificar que todas estén incluidas
print("path_lengths keys:", list(path_lengths.keys()))

# Definir el ancho de línea para cada caso
line_widths = {"BM_3": 2, "BM_6": 8, "CRW_5": 2, "CRW_6": 2, "Levy_1": 2, "Levy_07": 2}

# Crear la figura
fig = go.Figure()

# Agregar trazas en un loop
for key, df in walks.items():
    if df is None:
        print(f"Advertencia: La caminata '{key}' es None y no se graficará.")
        continue  # Saltar esta iteración si df es None

    if key not in path_lengths:
        print(f"Advertencia: '{key}' no está en path_lengths y no se graficará.")
        continue  # Evitar error si path_lengths[key] no existe

    fig.add_trace(go.Scatter(
        x=np.arange(len(path_lengths[key])),
        y=path_lengths[key],
        marker=dict(size=2),
        line=dict(width=line_widths.get(key, 2)),  # Usar un ancho de línea por defecto si falta
        mode='lines',
        name=f'Path length {key.replace("_", " ")}',
        showlegend=True
    ))

# Configuración del layout
fig.update_layout(
    title_text='Path length - (BM1 vs BM2 vs CRW)',
    autosize=False,
    width=900,
    height=500
)

# Mostrar la gráfica
fig.show()



Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(0.6713619907026533), np.float64(1.146233526693129)), (np.float64(4.044274792008752), np.float64(13.845840747258508)), (np.float64(12.964542525816416), np.float64(14.819767673370764)), (np.float64(12.973843686000338), np.float64(14.785019708850285))]
Cantidad de puntos generados: 1001


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(1.0208948018588169), np.float64(-2.0640470530513713)), (np.float64(1.1596908453056873), np.float64(-2.1512723366457274)), (np.float64(30.65171775920214), np.float64(-14.741075101965151)), (np.float64(30.919995075198404), np.float64(-15.001058894522911))]
Cantidad de puntos generados: 1001


walks keys: ['BM_3', 'BM_6', 'CRW_5', 'CRW_6', 'Levy_1', 'Levy_07']


ValueError: array must not contain infs or NaNs

### Análisis de Longitud de Trayectoria
Se compara la longitud del camino recorrido en diferentes tipos de caminatas.

# Activity 2: Lévy Distribution - N Different Curves

In [ ]:
#############################
# Mean Squared Displacement #
#############################
def msd(trajectory):
    """
    Compute the Mean Squared Displacement (MSD) for a given trajectory.
    
    Parameters:
    trajectory: A numpy array containing the trajectory
    
    Returns:
    msd: The MSD as a function of time.
    """
    N = len(trajectory)    
    msd = np.zeros(N-1)
    
    for i in range(1,N):
        displacements = trajectory[i:] - trajectory[:N - i]
        squared_displacements = np.sum(displacements**2, axis=1) # Square each coordinate of the displacement and add them together
        msd[i-1] = np.mean(squared_displacements) # Save the msd
    
    return msd

n_steps = 1000

# Definir las configuraciones de cada tipo de caminata
walks = {
    "BM_3": bm_2d(n_steps, speed=3),
    "BM_6": bm_2d(n_steps, speed=6),
    "CRW_6_c0.6": rw_2d(n_steps, speed=6, c=0.6),
    "CRW_6_c0.9": rw_2d(n_steps, speed=6, c=0.9),
    "Levy_6_alpha1": levy_flight(n_steps, alpha=1, c=0.5),
    "Levy_6_alpha0.7": levy_flight(n_steps, alpha=0.7, c=0.5)
}
# Verificar si alguna caminata devolvió None
for key, df in walks.items():
    if df is None:
        print(f"Error: La función para '{key}' devolvió None.")

# Obtener las trayectorias (solo columnas x_pos y y_pos)
trajectories = {
    key: df[['x_pos', 'y_pos']].values for key, df in walks.items() if isinstance(df, pd.DataFrame) and not df.empty
}

# Calcular MSD para cada trayectoria
msd_values = {key: msd(traj) for key, traj in trajectories.items()}

# Crear la figura
fig = go.Figure()

# Agregar trazas en un loop
for key, df in walks.items():
    if df is None:
        print(f"Advertencia: {key} es None y no se graficará.")
        continue  # Saltar esta iteración si df es None
    
    fig.add_trace(go.Scatter(
        x=df.index,
        y=msd_values.get(key, []),  # Evitar error si key no está en msd_values
        marker=dict(size=2),
        line=dict(width=2),
        mode='lines',
        name=f'MSD {key.replace("_", " ")}',
        showlegend=True
    ))

# Configuración del layout
fig.update_layout(
    title_text="Mean Squared Displacement - (BM vs CRW)",
    autosize=False,
    width=900,
    height=500,
    xaxis=dict(title="Time Step"),
    yaxis=dict(title="Mean Squared Displacement")
)

# Mostrar la gráfica
fig.show()

Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(4.848628438768356), np.float64(-2.409975551797226)), (np.float64(5.020504069015049), np.float64(-2.7385964134723335)), (np.float64(4.212222826006189), np.float64(-3.2809595883346026)), (np.float64(2.082455522194726), np.float64(-3.224305975390078))]
Cantidad de puntos generados: 1001


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(0.13203566919526352), np.float64(-0.015984564350612393)), (np.float64(1.4315027587034943), np.float64(2.4700878472535868)), (np.float64(6.640441549665935), np.float64(-0.30362304532680096)), (np.float64(6.760138379811756), np.float64(-0.43000311799531166))]
Cantidad de puntos generados: 1001


Error: La función para 'Levy_6_alpha1' devolvió None.
Error: La función para 'Levy_6_alpha0.7' devolvió None.
Advertencia: Levy_6_alpha1 es None y no se graficará.
Advertencia: Levy_6_alpha0.7 es None y no se graficará.


### Análisis de Desplazamiento Cuadrático Medio (MSD)
Se calcula el desplazamiento cuadrático medio para analizar la relación entre el tiempo y la distancia recorrida en cada tipo de caminata.

# Activity 3: Histograms + Curves


In [ ]:

def trajectory_angles(trajectory):    
    angles = []

    for i in range(1, len(trajectory) - 1):
        # Get Vectors
        V1 = np.array(trajectory.iloc[i] - trajectory.iloc[i - 1]) # v(i) - v(i-1)
        V2 = np.array(trajectory.iloc[i + 1] - trajectory.iloc[i]) # v(i+1) - v(i)  
        
        # Dot Product
        dot_product = np.dot(V1, V2)
        
        # Vector magnitudes
        norm_V1 = np.linalg.norm(V1)
        norm_V2 = np.linalg.norm(V2)
        
        # Avoid div by 0
        if norm_V1 == 0 or norm_V2 == 0:
            angles.append(0)
            continue
        
        # Calculate the angle     
        cos_theta = dot_product / (norm_V1 * norm_V2)        
                
        theta = np.arccos(cos_theta)  # Cos -1  get angles on [0, 2π]

        # Convert V1 and V2 to 3D (to apply the cross product)
        V1_3D = np.array([V1[0], V1[1], 0], dtype=np.float64)
        V2_3D = np.array([V2[0], V2[1], 0], dtype=np.float64)
        
        # Calculate the cross product in 3D to determine the sign of the angle
        cross_product = np.cross(V1_3D, V2_3D)[-1]
        
        # If cross product is negative, the angle is negative
        if cross_product < 0:
            theta = -theta            
        
        angles.append(theta)  
    
    return pd.Series(angles)

cauchy1 = 0.4
cauchy2 = 0.7
n_steps = 1000
# Generate the trajectories
crw_6_c1 = rw_2d(n_steps, s_pos=[2,5], c=cauchy1)
crw_6_c2 = rw_2d(n_steps, s_pos=[2,5], c=cauchy2)

# Calculate the angles
crw_6_c1['angle'] = trajectory_angles(crw_6_c1).shift(1)  
crw_6_c2['angle'] = trajectory_angles(crw_6_c2).shift(1)

# Plot the results
resolution = 500
aux_domain = np.linspace(-np.pi, np.pi, resolution)

aux = np.mod(aux_domain, 2 * np.pi)
aux2 = np.mod(aux_domain, 2 * np.pi)

figuraWrapc_pdf = go.Figure()
wrapcauchy_1_pdf = np.array([wrapcauchy.pdf(i,cauchy1) for i in aux])
wrapcauchy_2_pdf = np.array([wrapcauchy.pdf(i,cauchy2) for i in aux2])

figuraWrapc_pdf.add_trace(go.Histogram(
    x=crw_6_c1.angle.values, 
    nbinsx=150, 
    histnorm='probability density', 
    marker=dict(color='red'),
    opacity=0.5,
    name=f'Observed Cauchy {cauchy1}'
))
figuraWrapc_pdf.add_trace(go.Histogram(
    x=crw_6_c2.angle.values, 
    nbinsx=150, 
    histnorm='probability density',
    marker=dict(color='blue'),
    opacity=0.5,
    name=f'Observed Cauchy {cauchy2}'
))
figuraWrapc_pdf.add_trace(
    go.Scatter(
        x = aux_domain,
        y = wrapcauchy_1_pdf,
        marker = dict(size=2),
        line = dict(width=2, color='red'),
        mode = 'lines',
        name=f'CRW, Cauchy {cauchy1}',
        showlegend = True
    )
)
figuraWrapc_pdf.add_trace(
    go.Scatter(
        x = aux_domain,
        y = wrapcauchy_2_pdf,
        marker = dict(size=2),
        line = dict(width=2,  color='blue'),
        mode = 'lines',            
        name=f'CRW, Cauchy {cauchy2}',
        showlegend = True
    )
)

figuraWrapc_pdf.update_layout(
    title_text = 'Turning-angle Distribution - (source dist. vs observed dist.) ',
    autosize = False,
    width=1000, 
    height=500,
    xaxis=dict(title="Turning Angle"),  # Set X-axis label
    yaxis=dict(title="Probability Density")  # Set Y-axis label
)

figuraWrapc_pdf.show()

# Activity 4:  Step-length Distribution

In [31]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import levy_stable, wrapcauchy



# Parámetros iniciales
alpha_values = [0.9, 0.6]
m = 4.5  # Location parameter
beta = 1
n_steps = 1000
resolution = 600
x = np.linspace(0, 50, resolution)

# Generar las caminatas de Lévy y calcular las longitudes de los pasos
levy_walks = [levy_flight(n_steps, alpha=alpha, scale=1.0) for alpha in alpha_values]
step_lengths = [lw["step_length"].dropna()[lw["step_length"] <= 50] for lw in levy_walks]
# step_lengths = [lw[lw <= 50] for lw in levy_walks]  # Filtrar valores ≤ 50

# Crear la figura
fig_levy_pdf = go.Figure()

# # Agregar histogramas de longitudes de paso observadas
# colors = ['blue', 'red']
# for sl, alpha, color in zip(step_lengths, alpha_values, colors):
#     fig_levy_pdf.add_trace(go.Histogram(
#         x=sl -m,
#         nbinsx=80,
#         histnorm='probability density',
#         marker=dict(color=color),
#         opacity=0.5,
#         name=f'Observed α = {alpha}'
#     ))
# Agregar histogramas de longitudes de paso observadas
colors = ['blue', 'red']
for sl, alpha, color in zip(step_lengths, alpha_values, colors):
    shift = m if alpha == 0.6 else 0  # Ajustar solo para α = 0.6
    fig_levy_pdf.add_trace(go.Histogram(
        x=sl + shift,  # Aplicar el desplazamiento adecuado
        nbinsx=80,
        histnorm='probability density',
        marker=dict(color=color),
        opacity=0.5,
        name=f'Observed α = {alpha}'
    ))
    # no pude hacer que el histograma se ajustara a la curva de la distrubucion de levi
# Agregar curvas de la distribución estable de Lévy
for alpha, color in zip(alpha_values, colors):
    fig_levy_pdf.add_trace(go.Scatter(
        x=x,
        y=levy_stable.pdf(x, alpha, beta, loc=m),
        line=dict(width=2, color=color),
        mode='lines',
        name=f'Levy α = {alpha}'
    )) 

# Configuración del gráfico
fig_levy_pdf.update_layout(
    title="Step-length Distribution - (Source vs Observed)",
    width=800, 
    height=500,
    xaxis=dict(title="Step Length", range=[0, 50]),
    yaxis=dict(title="Probability Density"),
)

fig_levy_pdf.show()


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(5.957025625443614), np.float64(-8.121590640410407)), (np.float64(6.593476367727975), np.float64(-10.053450186622413)), (np.float64(7.325471327119742), np.float64(-10.248138039446104)), (np.float64(5.981320758714828), np.float64(-10.496515346945575))]
Cantidad de puntos generados: 1001


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(0.1512462299319315), np.float64(-0.1305466063540138)), (np.float64(-0.21698763774764612), np.float64(-0.7602972413951823)), (np.float64(-0.6146407702123786), np.float64(-1.9822974159706392)), (np.float64(-0.7286052193055816), np.float64(-4.106876625164296))]
Cantidad de puntos generados: 1001
